# Taxonomic Backbone Analysis
Analysis of WFO (World Flora Online) classification data with UBC Botanical Garden collection coverage.

## Setup

In [1]:
import pandas as pd
import numpy as np

## Load Data

In [2]:
# Load WFO classification
wfo_path = "data/classification.csv"
df = pd.read_csv(
    wfo_path,
    sep="\t",
    encoding="ISO-8859-1",
    dtype=str,
    low_memory=False
)

# Load UBC checklist
ubc_path = "data/2025_unique-taxa-checklist_ubc.csv"
ubc = pd.read_csv(ubc_path, dtype=str)

print(f"WFO shape: {df.shape}")
print(f"UBC shape: {ubc.shape}")

WFO shape: (1657866, 29)
UBC shape: (12822, 3)


## Explore WFO Data

In [3]:
print("Taxonomic Status:")
print(df["taxonomicStatus"].value_counts(dropna=False))

print("\nTaxon Rank (top 10):")
print(df["taxonRank"].value_counts(dropna=False).head(10))

Taxonomic Status:
taxonomicStatus
Synonym      1022626
Accepted      453167
Unchecked     182073
Name: count, dtype: int64

Taxon Rank (top 10):
taxonRank
species       1193834
variety        267952
subspecies      82287
form            51060
genus           45389
section          3809
subvariety       2596
unranked         2037
subgenus         1298
family           1277
Name: count, dtype: int64


## Worldwide Statistics

In [4]:
# Filter to accepted species with family and genus
accepted_species = df[
    (df["taxonomicStatus"] == "Accepted") &
    (df["taxonRank"] == "species") &
    (df["family"].notna()) &
    (df["genus"].notna())
].copy()

print(f"Accepted species: {accepted_species.shape[0]}")

Accepted species: 381467


In [5]:
# Genera per family
genera_per_family = (
    accepted_species
    .groupby("family")["genus"]
    .nunique()
    .sort_values(ascending=False)
)

# Species per genus
species_per_genus = (
    accepted_species
    .groupby(["family", "genus"])
    .size()
    .sort_values(ascending=False)
    .rename("n_species")
)

print(f"Total families: {genera_per_family.shape[0]}")
print(f"\nFamily with most genera:\n{genera_per_family.head(1)}")
print(f"\nTop 10 genera by species count:\n{species_per_genus.head(10)}")

# Distribution stats
dist = species_per_genus.reset_index()["n_species"]
print(f"\nSpecies per genus distribution:")
print(dist.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"\nMonospecific genera: {(dist == 1).sum()}")

Total families: 734

Family with most genera:
family
Asteraceae    1717
Name: genus, dtype: int64

Top 10 genera by species count:
family           genus       
Asteraceae       Hieracium       3235
Fabaceae         Astragalus      3116
Asteraceae       Taraxacum       2600
Piperaceae       Piper           2436
Cyperaceae       Carex           2351
Orchidaceae      Bulbophyllum    2191
Begoniaceae      Begonia         2191
Euphorbiaceae    Euphorbia       2105
Melastomataceae  Miconia         1938
Orchidaceae      Epidendrum      1869
Name: n_species, dtype: int64

Species per genus distribution:
count    15755.000000
mean        24.212440
std         98.212172
min          1.000000
50%          4.000000
75%         14.000000
90%         47.000000
95%         98.000000
99%        333.460000
max       3235.000000
Name: n_species, dtype: float64

Monospecific genera: 4546


## Including Infraspecific Taxa

In [6]:
# Include subspecies, varieties, forms, subvarieties
accepted_taxa = df[
    (df["taxonomicStatus"] == "Accepted") &
    (df["taxonRank"].isin(["species", "subspecies", "variety", "form", "subvariety"])) &
    (df["family"].notna()) &
    (df["genus"].notna())
].copy()

taxa_per_genus = (
    accepted_taxa
    .groupby(["family", "genus"])
    .size()
    .sort_values(ascending=False)
    .rename("n_taxa")
)

print("Top 10 genera by taxa count (including infraspecific):")
print(taxa_per_genus.head(10))

Top 10 genera by taxa count (including infraspecific):
family           genus       
Asteraceae       Hieracium       6246
Fabaceae         Astragalus      3607
Cyperaceae       Carex           2836
Asteraceae       Taraxacum       2619
Piperaceae       Piper           2598
Begoniaceae      Begonia         2502
Euphorbiaceae    Euphorbia       2489
Orchidaceae      Bulbophyllum    2343
Rosaceae         Rubus           2016
Melastomataceae  Miconia         1980
Name: n_taxa, dtype: int64


## UBC Collection Analysis

In [7]:
# Get UBC families
ubc_families = (
    ubc["Family"]
    .dropna()
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .unique()
)

# Filter WFO to UBC families
accepted_species_ubc = accepted_species[accepted_species["family"].isin(ubc_families)]

genera_per_family_ubc = (
    accepted_species_ubc
    .groupby("family")["genus"]
    .nunique()
    .sort_values(ascending=False)
)

species_per_genus_ubc = (
    accepted_species_ubc
    .groupby(["family", "genus"])
    .size()
    .sort_values(ascending=False)
    .rename("n_species")
)

print(f"UBC families matched: {len(ubc_families)}")
print(f"\nTop 20 families by genera:\n{genera_per_family_ubc.head(20)}")
print(f"\nTop 20 genera by species:\n{species_per_genus_ubc.head(20)}")

UBC families matched: 316

Top 20 families by genera:
family
Asteraceae         1717
Poaceae             814
Fabaceae            807
Orchidaceae         745
Rubiaceae           611
Apiaceae            450
Apocynaceae         381
Brassicaceae        361
Malvaceae           245
Lamiaceae           231
Euphorbiaceae       228
Acanthaceae         205
Arecaceae           186
Cactaceae           185
Melastomataceae     164
Gesneriaceae        161
Asparagaceae        159
Rutaceae            151
Araceae             143
Sapindaceae         141
Name: genus, dtype: int64

Top 20 genera by species:
family           genus       
Asteraceae       Hieracium       3235
Fabaceae         Astragalus      3116
Asteraceae       Taraxacum       2600
Piperaceae       Piper           2436
Cyperaceae       Carex           2351
Begoniaceae      Begonia         2191
Orchidaceae      Bulbophyllum    2191
Euphorbiaceae    Euphorbia       2105
Melastomataceae  Miconia         1938
Orchidaceae      Epidendrum      1

## UBC Coverage Analysis

Calculate what percentage of each genus's global diversity UBC has grown.

In [8]:
# Count UBC species per genus
ubc_clean = ubc.copy()
ubc_clean["Family"] = ubc_clean["Family"].str.strip()
ubc_clean["Genus"] = ubc_clean["Genus"].str.strip()
ubc_clean["Species"] = ubc_clean["Species"].str.strip()

ubc_counts = (
    ubc_clean
    .groupby(["Family", "Genus"])
    .agg(n_species_at_ubc=("Species", "nunique"))
    .reset_index()
    .rename(columns={"Family": "family", "Genus": "genus"})
)

# Get WFO worldwide counts
wfo_counts = (
    accepted_species
    .groupby(["family", "genus"])
    .size()
    .reset_index(name="n_species_worldwide")
)

# Merge and calculate coverage
coverage = ubc_counts.merge(wfo_counts, on=["family", "genus"], how="left")
coverage["percent_coverage"] = (
    (coverage["n_species_at_ubc"] / coverage["n_species_worldwide"]) * 100
).round(2)

# Flag issues
coverage["needs_review"] = coverage["percent_coverage"] > 100
coverage["is_monotypic"] = coverage["n_species_worldwide"] == 1

# Adjusted coverage (cap monotypic at 100%)
coverage["percent_adjusted"] = coverage["percent_coverage"]
mask = coverage["is_monotypic"]
coverage.loc[mask, "percent_adjusted"] = coverage.loc[mask, "percent_adjusted"].clip(upper=100)

# Summary
print(f"Total genera in UBC: {len(coverage)}")
print(f"Genera needing review (>100%): {coverage['needs_review'].sum()}")
print(f"Monotypic genera: {coverage['is_monotypic'].sum()}")

# Valid genera only
valid = coverage[~coverage["needs_review"]].sort_values("percent_adjusted", ascending=False)
print(f"\nTop 20 genera by coverage (valid only):")
print(valid[["family", "genus", "n_species_at_ubc", "n_species_worldwide", 
             "percent_adjusted", "is_monotypic"]].head(20))

print(f"\nMean coverage: {valid['percent_adjusted'].mean():.2f}%")
print(f"Median coverage: {valid['percent_adjusted'].median():.2f}%")

# Review list
review = coverage[coverage["needs_review"]].sort_values("percent_coverage", ascending=False)
print(f"\nGenera needing taxonomic review (top 10):")
print(review[["family", "genus", "n_species_at_ubc", "n_species_worldwide", 
              "percent_coverage"]].head(10))

Total genera in UBC: 2278
Genera needing review (>100%): 32
Monotypic genera: 211

Top 20 genera by coverage (valid only):
                family               genus  n_species_at_ubc  \
1212    Hamamelidaceae        Sinowilsonia                 1   
622   Brachytheciaceae  Pseudoscleropodium                 1   
615       Boraginaceae        Trachystemon                 1   
612       Boraginaceae          Pontechium                 1   
1627      Philesiaceae           Lapageria                 1   
610       Boraginaceae        Pentaglottis                 1   
1637          Pinaceae             Cathaya                 1   
1643          Pinaceae         Pseudolarix                 1   
1652    Plantaginaceae             Asarina                 1   
1660    Plantaginaceae      Ellisiophyllum                 1   
1664    Plantaginaceae         Hemiphragma                 1   
570       Bignoniaceae           Chilopsis                 1   
1669    Plantaginaceae          Melosperma   

## Export Worldwide Data

In [16]:
# Prepare worldwide dataset
worldwide_export = (
    accepted_species
    .groupby(["family", "genus"], dropna=False)
    .size()
    .reset_index(name="n_species_in_genus")
)

family_stats = (
    worldwide_export
    .groupby("family", dropna=False)
    .agg(
        n_genera_in_family=("genus", "nunique"),
        n_species_in_family=("n_species_in_genus", "sum")
    )
    .reset_index()
)

worldwide_tidy = (
    worldwide_export
    .merge(family_stats, on="family", how="left")
    .sort_values(["family", "genus"])
)

print(f"Worldwide dataset: {worldwide_tidy.shape[0]} rows")


Worldwide dataset: 15755 rows


### Save Worldwide Dataset

**Run the cell below to export the worldwide processed backbone.**

In [10]:
worldwide_tidy.to_csv("./wfo_worldwide_family_genus.csv", index=False, encoding="utf-8")
print("✓ Exported: wfo_worldwide_family_genus.csv")

✓ Exported: wfo_worldwide_family_genus.csv


## Export the UBC coverage data

In [17]:
# Prepare UBC coverage dataset
coverage_export = coverage.sort_values(["family", "genus"])

print(f"UBC coverage dataset: {coverage_export.shape[0]} rows")
print(coverage_export.head(10))

UBC coverage dataset: 2278 rows
          family           genus  n_species_at_ubc  n_species_worldwide  \
0    Acanthaceae        Acanthus                 7                 30.0   
1    Acanthaceae         Ruellia                 2                380.0   
2    Acanthaceae   Strobilanthes                 3                459.0   
3    Acanthaceae      Thunbergia                 1                151.0   
4      Acoraceae          Acorus                 3                  2.0   
5  Actinidiaceae       Actinidia                14                 57.0   
6  Actinidiaceae  Clematoclethra                 2                  1.0   
7  Actinidiaceae        Saurauia                 2                392.0   
8      Aizoaceae      Aloinopsis                 2                  8.0   
9      Aizoaceae    Bergeranthus                 2                 10.0   

   percent_coverage  needs_review  is_monotypic  percent_adjusted  
0             23.33         False         False             23.33  
1     

### Save UBC Coverage Dataset

**Run this cell to export the UBC coverage analysis (includes taxonomic review flags).**

In [15]:
coverage_export.to_csv("./ubc_coverage_analysis.csv", index=False, encoding="utf-8")
print("✓ Exported: ubc_coverage_analysis.csv")

✓ Exported: ubc_coverage_analysis.csv
